# Progress Evaluation 2 - Four-model comparison

This is a fully standalone notebook. Run each cell from top to bottom.

In [4]:
from pathlib import Path
from time import perf_counter
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, ConfusionMatrixDisplay,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_DIR, RESULTS_DIR, MODELS_DIR = Path('dataset'), Path('results'), Path('models')
RESULTS_DIR.mkdir(exist_ok=True); MODELS_DIR.mkdir(exist_ok=True)
X_train = pd.read_csv(DATA_DIR / 'X_train.csv').astype(float)
X_test = pd.read_csv(DATA_DIR / 'X_test.csv').astype(float)
y_train = pd.read_csv(DATA_DIR / 'y_train.csv').squeeze('columns').astype(int)
y_test = pd.read_csv(DATA_DIR / 'y_test.csv').squeeze('columns').astype(int)
assert list(X_train.columns) == list(X_test.columns)
assert not X_train.isna().any().any() and not X_test.isna().any().any()
print('Train:', X_train.shape, 'Test:', X_test.shape)
display(y_train.value_counts(normalize=True).rename('class proportion').round(3))

Train: (25217, 47) Test: (6305, 47)


loan_status
0    0.784
1    0.216
Name: class proportion, dtype: float64

## Algorithms, validation, and tuning

SMOTE and feature selection are inside the pipeline. Therefore, during stratified CV, synthetic samples and selected features are created from the training fold only; this prevents validation leakage. ROC-AUC is used for tuning because the loan-default target is imbalanced.

In [5]:
def make_pipeline(model):
    return Pipeline([('smote', SMOTE(random_state=RANDOM_STATE)),
                     ('select', SelectKBest(f_classif, k='all')),
                     ('model', model)])

feature_options = {'select__k': ['all', 20, 30]}
models_to_compare = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2000, solver='liblinear', random_state=RANDOM_STATE),
        feature_options | {'model__C': np.logspace(-3, 2, 10).tolist(), 'model__class_weight': [None, 'balanced']}
    ),
    'K-Nearest Neighbours': (
        KNeighborsClassifier(),
        feature_options | {'model__n_neighbors': [5, 11, 15, 21, 31], 'model__weights': ['uniform', 'distance'], 'model__metric': ['euclidean', 'manhattan']}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
        feature_options | {'model__n_estimators': [200, 300, 400], 'model__max_depth': [None, 8, 12, 16, 24], 'model__min_samples_split': [2, 5, 10], 'model__min_samples_leaf': [1, 2, 4], 'model__max_features': ['sqrt', 'log2']}
    ),
    'XGBoost': (
        XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=1, tree_method='hist'),
        feature_options | {'model__n_estimators': [150, 250, 350], 'model__max_depth': [3, 4, 5, 6], 'model__learning_rate': [0.03, 0.05, 0.1, 0.2], 'model__subsample': [0.7, 0.85, 1.0], 'model__colsample_bytree': [0.7, 0.85, 1.0]}
    ),
}
print('Models:', list(models_to_compare))

Models: ['Logistic Regression', 'K-Nearest Neighbours', 'Random Forest', 'XGBoost']


In [6]:
# Use QUICK_RUN=True only for a short test. Keep False for PE2 evidence.
QUICK_RUN = False
cv = StratifiedKFold(n_splits=3 if QUICK_RUN else 5, shuffle=True, random_state=RANDOM_STATE)
iterations = 2 if QUICK_RUN else 12
scoring = {'roc_auc': 'roc_auc', 'f1': 'f1', 'recall': 'recall', 'precision': 'precision'}
rows, tuning, fitted_models = [], [], {}

for name, (model, parameters) in models_to_compare.items():
    print('\n' + '=' * 60 + '\n' + name)
    baseline = cross_validate(make_pipeline(clone(model)), X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    search = RandomizedSearchCV(make_pipeline(clone(model)), parameters, n_iter=iterations, scoring='roc_auc', cv=cv, n_jobs=-1, random_state=RANDOM_STATE, refit=True)
    started = perf_counter(); search.fit(X_train, y_train); elapsed = perf_counter() - started
    best = search.best_estimator_; fitted_models[name] = best
    predicted, probability = best.predict(X_test), best.predict_proba(X_test)[:, 1]
    row = {'model': name, 'baseline_cv_roc_auc': baseline['test_roc_auc'].mean(),
           'tuned_cv_roc_auc': search.best_score_, 'cv_roc_auc_std': search.cv_results_['std_test_score'][search.best_index_],
           'test_roc_auc': roc_auc_score(y_test, probability), 'test_average_precision': average_precision_score(y_test, probability),
           'test_accuracy': accuracy_score(y_test, predicted), 'test_precision': precision_score(y_test, predicted, zero_division=0),
           'test_recall': recall_score(y_test, predicted, zero_division=0), 'test_f1': f1_score(y_test, predicted, zero_division=0),
           'selected_features': int(best.named_steps['select'].get_support().sum()), 'search_seconds': elapsed}
    rows.append(row); tuning.append({'model': name, 'best_parameters': search.best_params_, 'best_cv_roc_auc': search.best_score_})
    print('Best parameters:', search.best_params_)
    print(f"Test ROC-AUC: {row['test_roc_auc']:.4f}")


Logistic Regression
Best parameters: {'select__k': 'all', 'model__class_weight': None, 'model__C': 0.01291549665014884}
Test ROC-AUC: 0.8554

K-Nearest Neighbours
Best parameters: {'select__k': 30, 'model__weights': 'uniform', 'model__n_neighbors': 21, 'model__metric': 'manhattan'}
Test ROC-AUC: 0.8527

Random Forest
Best parameters: {'select__k': 'all', 'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 4, 'model__max_features': 'sqrt', 'model__max_depth': None}
Test ROC-AUC: 0.9035

XGBoost
Best parameters: {'select__k': 30, 'model__subsample': 0.85, 'model__n_estimators': 350, 'model__max_depth': 4, 'model__learning_rate': 0.2, 'model__colsample_bytree': 1.0}
Test ROC-AUC: 0.9256


In [8]:
comparison = pd.DataFrame(rows).sort_values('test_roc_auc', ascending=False).reset_index(drop=True)
comparison.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False, float_format='%.4f')
with open(RESULTS_DIR / 'tuning_summary.json', 'w', encoding='utf-8') as file: json.dump(tuning, file, indent=2, default=str)
final_name = comparison.loc[0, 'model']
joblib.dump(fitted_models[final_name], MODELS_DIR / 'final_credit_risk_model.joblib')
with open(MODELS_DIR / 'final_model_selection.txt', 'w', encoding='utf-8') as file:
    file.write(f'Selected model: {final_name}\nSelection rule: highest independent test ROC-AUC after CV tuning.\n')
display(comparison.round(4))
print('Final selected model:', final_name)

,model,baseline_cv_roc_auc,tuned_cv_roc_auc,cv_roc_auc_std,test_roc_auc,test_average_precision,test_accuracy,test_precision,test_recall,test_f1,selected_features,search_seconds
0,XGBoost,0.9208,0.9217,0.0049,0.9256,0.8562,0.9075,0.8845,0.6579,0.7545,30,56.9615
1,Random Forest,0.8978,0.8998,0.0068,0.9035,0.8229,0.8967,0.8319,0.6542,0.7324,47,344.9675
2,Logistic Regression,0.8505,0.8512,0.0043,0.8554,0.6645,0.7792,0.4929,0.7592,0.5977,47,27.5026
3,K-Nearest Neighbours,0.7720,0.8404,0.0083,0.8527,0.6744,0.7851,0.5017,0.7577,0.6037,30,88.7147


Final selected model: XGBoost
